# PER with a Sparse Pauli–Lindblad Noise Model 

This notebook is a **theory-driven** introduction to **Probabilistic Error Reduction / Cancellation (PER/PEC)** using a **sparse Pauli–Lindblad noise model**, following:

E. van den Berg, Z. K. Minev, A. Kandala, K. Temme  
*Probabilistic error cancellation with sparse Pauli–Lindblad model* (arXiv:2201.09866)

We work through a **single, concrete example** end-to-end:
a 4-qubit VQE circuit (HVA for a 2×2 mixed-field Ising model), executed on a realistic noisy backend (Fake IBM snapshot).

---

The steps:

1. **Construct a target circuit** (VQE circuit from the previous notebook)
2. **Model noise as quantum channels** and motivate *layer-wise* noise
3. **Introduce the sparse Pauli–Lindblad model** (compact support, sparse topology)
4. **Learn the model** via tomography primitives:
   - Pauli twirling → diagonal Pauli channel
   - fidelity pairs → SPAM-robust decay
   - pair-breaking → split paired fidelities
5. **Apply PER** by sampling a partial inverse at several noise-strength values, and
6. **(Optional)** combine with ZNE to extrapolate to the zero-noise limit.

Throughout, we will explicitly state:
- what is assumed,
- what is measured,
- what is reconstructed,
- and where approximations enter.


## 1. Running example: a 4-qubit VQE circuit (HVA for MFIM)

To keep continuity, we reuse the same 4-qubit circuit family as in the previous notebook.
We will:
- build the MFIM Hamiltonian,
- build a 1-layer HVA ansatz,
- bind optimized parameters,
- and use this circuit as the target workload for tomography and PER.

The circuit itself is not the focus here — it is simply a realistic NISQ-style circuit with
single- and two-qubit gates and nontrivial depth.

In [137]:
import sys

sys.path.append('../..')
import numpy as np

topology = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])

from src.VQE_functions import optimize_energy, MFIM_Hamiltonian, build_hva_layers
from scipy.linalg import eigh
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, SparsePauliOp

topology = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])

hx = 1
hz = 1
J = -1

Hamiltonian = MFIM_Hamiltonian(J=J, hx=hx, hz=hz, topology = topology)
num_qubits = Hamiltonian.num_qubits
pauli_list = [p.to_label() for p in Hamiltonian.paulis]
hva_layers, params = build_hva_layers(topology=topology, num_layers=1)

backend = AerSimulator()

num_layers = 1
initial_params = [1,1,1]
bounds = [(-np.pi, np.pi)] * 3*num_layers

opt_params, energy, HelperInfo = optimize_energy(initial_params=initial_params, Hamiltonian=Hamiltonian, 
                                                 topology=topology, num_layers=num_layers, backend=backend, 
                                                 mode="measurement",bounds = bounds, shots=1024)

values = {p: v for p, v in zip(params, opt_params)}
qc_bound = hva_layers.assign_parameters(values)

print(qc_bound.draw(fold=-1))

exact_energy = np.min(eigh(Hamiltonian.to_matrix(), eigvals_only=True))

psi = Statevector.from_instruction(qc_bound)
expectation_values = []
for pauli_str in pauli_list:
    op = SparsePauliOp.from_list([(pauli_str, 1.0)])
    expval = np.real(psi.expectation_value(op))
    expectation_values.append(expval)

print("Measured energy: ",energy)
print("Relative error: ", np.abs(energy - exact_energy) / np.abs(exact_energy) *100, "%")
print("Initial parameters: ",initial_params)
print("Optimal parameters: ", opt_params)

     ┌───┐┌─────────────┐┌────────────┐                                                                   
q_0: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├──■──────────────────■────■────────────────────────────────■───────
     ├───┤├─────────────┤├────────────┤┌─┴─┐┌────────────┐┌─┴─┐  │                                │       
q_1: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├┤ X ├┤ Rz(1.5871) ├┤ X ├──┼────────■───────────────────────┼────■──
     ├───┤├─────────────┤├────────────┤└───┘└────────────┘└───┘  │      ┌─┴─┐     ┌────────────┐  │  ┌─┴─┐
q_2: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├──■──────────────────■────┼──────┤ X ├─────┤ Rz(1.5871) ├──┼──┤ X ├
     ├───┤├─────────────┤├────────────┤┌─┴─┐┌────────────┐┌─┴─┐┌─┴─┐┌───┴───┴────┐└────────────┘┌─┴─┐└───┘
q_3: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├┤ X ├┤ Rz(1.5871) ├┤ X ├┤ X ├┤ Rz(1.5871) ├──────────────┤ X ├─────
     └───┘└─────────────┘└────────────┘└───┘└────────────┘└───┘└───┘└────────────┘              └───┘     
Measured energy:  -8.576171875
Relati


We will use this fixed circuit `qc` as the object of study.

The quantity of interest in VQE is the energy

$$
E = \langle H \rangle = \sum_{m} h_m \langle P_m \rangle
$$

i.e., a weighted sum of Pauli expectation values.

In an ideal setting, these expectation values are estimated from repeated measurements (“shots”).
On real hardware, the estimator becomes **biased by noise**, and PER aims to remove this bias
without changing the hardware.

## 2. Noise model: channels acting on density matrices

A quantum circuit ideally implements a unitary channel
$$
\mathcal{U}(\rho) = U \rho U^\dagger.
$$

On real hardware, each operation is followed (or accompanied) by noise.
A standard abstraction is to describe noise as a **completely positive trace-preserving (CPTP)** map
(a quantum channel) $\Lambda$, such that the implemented operation becomes

$$
\rho \mapsto \Lambda\!\left(\mathcal{U}(\rho)\right).
$$

If a circuit consists of gates $U_1, U_2, \dots, U_L$, the noisy circuit implements a composition
of alternating ideal gates and noise channels:

$$
\rho_{\text{out}}
= \Lambda_L \circ \mathcal{U}_L \circ \cdots \circ \Lambda_2 \circ \mathcal{U}_2 \circ \Lambda_1 \circ \mathcal{U}_1(\rho_{\text{in}}).
$$

In full generality, each $\Lambda_\ell$ could act nontrivially on **all qubits simultaneously**.
This is mathematically allowed but physically inconvenient:
the number of degrees of freedom of a general channel scales exponentially with system size.

To make progress, we exploit physical structure:
**NISQ circuits are built from 1- and 2-qubit gates**, and in current devices the dominant
errors typically come from **two-qubit entangling gates**.
This motivates a *layered* view of circuits.


## 3. Decompose the circuit into layers

Hardware-native compilation typically yields circuits built from:
- single-qubit rotations and
- entangling two-qubit gates.

Two key empirical facts motivate layer-wise modeling:

1. **Entangling gates are the main error source.**  
   On many superconducting devices, two-qubit gate error rates are commonly an order of magnitude
   larger than single-qubit gate errors (and often dominate the total infidelity).

2. **Two-qubit gates act locally along the device coupling graph.**  
   Qubits only interact if there is a physical link; routing through SWAPs creates additional
   noisy gates and changes the effective noise.

Because of this, we partition a compiled circuit into **layers** of the form:

- (A) parallel single-qubit gates, followed by
- (B) a set of two-qubit Clifford gates with **disjoint supports** (no qubit participates twice).

We then associate each layer $\ell$ with its own noise channel $\Lambda_\ell$ and write the total
noise as a concatenation of layer-wise channels:

$$
\Lambda_{\text{total}} \approx \Lambda_L \circ \Lambda_{L-1} \circ \cdots \circ \Lambda_1.
$$

This does *not* claim noise is truly independent between layers — it is an approximation that
becomes useful once combined with the next assumption: **sparsity**.


In [138]:
def circuit_to_layers(qc):
    """
    Split a circuit into benchmark layers of the form

        (any single-qubit gates)
        + (a disjoint set of two-qubit gates)

    A layer is kept only if it contains at least one two-qubit gate.
    """

    layers = []

    # remaining instructions (ignore measurements)
    inst_list = [ci for ci in qc.data if ci.operation.name != "measure"]

    # build layers until nothing is left
    while inst_list:

        circ = qc.copy_empty_like()
        layer_qubits = set()   # qubits occupied by 2Q gates in this layer

        # greedy sweep over remaining instructions
        for inst in inst_list.copy():

            support = set(inst.qubits)
            weight = len(inst.qubits)

            # add instruction if it does not overlap with existing 2Q supports
            if not layer_qubits.intersection(support):
                circ.append(inst.operation, inst.qubits, inst.clbits)
                inst_list.remove(inst)

                # 2Q gates block their qubits for further 2Q gates
                if weight == 2:
                    layer_qubits |= support

        # keep only layers that actually contain a 2Q gate
        for inst in circ.data:
            if len(inst.qubits) == 2:
                layers.append(circ)
                break

    return layers


def layers_with_barriers(qc, layers):
    """
    Reconstruct full circuit with barriers between benchmark layers.
    Each barrier is labeled with the layer index.
    """

    out = qc.copy_empty_like()

    for i, layer in enumerate(layers):
        out.compose(layer, inplace=True)

        # labeled barrier after each layer
        out.barrier(label=f"Layer {i}")

    return out



layers = circuit_to_layers(qc_bound)

qc_layers = layers_with_barriers(qc_bound, layers)

print(qc_layers.draw(fold=-1))


     ┌───┐┌─────────────┐┌────────────┐      Layer 0                     Layer 1            Layer 2                          Layer 3 
q_0: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├──■──────░────────────────────■──────░──────■───────────░────────────────────■───────────░────
     ├───┤├─────────────┤├────────────┤┌─┴─┐    ░    ┌────────────┐┌─┴─┐    ░      │           ░                    │           ░    
q_1: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├┤ X ├────░────┤ Rz(1.5871) ├┤ X ├────░──────┼────■──────░────────────────────┼────■──────░────
     ├───┤├─────────────┤├────────────┤└───┘    ░    └────────────┘└───┘    ░      │  ┌─┴─┐    ░    ┌────────────┐  │  ┌─┴─┐    ░    
q_2: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├──■──────░────────────────────■──────░──────┼──┤ X ├────░────┤ Rz(1.5871) ├──┼──┤ X ├────░────
     ├───┤├─────────────┤├────────────┤┌─┴─┐    ░    ┌────────────┐┌─┴─┐    ░    ┌─┴─┐└───┘    ░    ├────────────┤┌─┴─┐└───┘    ░    
q_3: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├┤ X ├────░────┤ Rz(1.58

### Pauli twirling → Pauli-diagonal effective noise

Pauli operators form a complete operator basis. For $n$ qubits, any density matrix can be written as
$$
\rho = \frac{1}{2^n}\sum_a r_a P_a,
\qquad P_a \in \{I,X,Y,Z\}^{\otimes n}.
$$
Since quantum channels are linear maps, their action on arbitrary states is fully determined
by how they act on the Pauli basis operators $P_a$.

A general noise channel acts as
$$
\Lambda(P_b) = \sum_a T_\Lambda[a,b]\,P_a,
$$
where $T_\Lambda$ is the Pauli transfer matrix. In general, this matrix is dense, meaning that
a single Pauli operator is mapped to a linear combination of many Pauli operators.

Pauli twirling removes this Pauli mixing by symmetrization. The twirled channel is defined as
$$
\Lambda(\cdot)
=
\frac{1}{|\mathcal P|}
\sum_{Q \in \mathcal P}
Q^\dagger\,\widetilde{\Lambda}\!\left(Q(\cdot)Q^\dagger\right)\,Q,
$$
where $\mathcal P$ is the Pauli group.
Averaging over all Pauli conjugations projects the channel onto the subspace that is invariant
under Pauli conjugation. As a result, the Pauli transfer matrix becomes diagonal and each Pauli
operator becomes an eigen-operator of the channel:
$$
\widetilde{\Lambda}_\ell(P_a) = f_{\ell,a}\,P_a.
$$
The coefficients
$$
f_{\ell,a}
=
2^{-n}\operatorname{Tr}\!\bigl(P_a\,\widetilde{\Lambda}_\ell(P_a)\bigr)
$$
are called **Pauli fidelities**.

---
#### Monte Carlo implementation

The exact average over all $4^n$ Pauli operators is not evaluated explicitly. Instead,
Pauli twirling is implemented via Monte Carlo sampling:

1. Sample a Pauli operator
   $$
   P = P_1 \otimes \cdots \otimes P_n,
   \qquad P_i \in \{I,X,Y,Z\},
   $$
   uniformly at random. Equivalently, each qubit independently chooses $I,X,Y,$ or $Z$
   with probability $1/4$.

2. Insert $P$ before the noisy layer.

3. Insert $P^\dagger$ after the noisy layer (or, if the layer is Clifford, insert the
   conjugated Pauli pushed through the Clifford).

4. Repeat this procedure for many circuit executions and average measurement outcomes.

In expectation, this Monte Carlo procedure converges to the exact Pauli-twirled channel.

---



In [139]:
import random
from qiskit.quantum_info import Pauli

PAULIS = ["I", "X", "Y", "Z"]

def apply_pauli_string(qc, pauli_string):
    """
    Apply a Pauli string given as ['I','X','Y','Z',...] to a circuit.
    """
    for q, p in enumerate(pauli_string):
        if p == "X":
            qc.x(q)
        elif p == "Y":
            qc.y(q)
        elif p == "Z":
            qc.z(q)

    return qc

def extract_two_qubit_structure(layer):
    """
    Return a list of (gate, q1, q2) for all 2Q gates in the layer,
    where gate is the actual Instruction (e.g. CXGate, CZGate).
    """
    structure = []

    for inst in layer.data:
        if len(inst.qubits) == 2:
            q1 = inst.qubits[0]._index
            q2 = inst.qubits[1]._index
            structure.append((inst.operation, q1, q2))

    return structure

def conjugate_pauli_2q(pauli_string, layer):
    """
    Conjugate Pauli string through ONLY the 2Q gates of `layer`
    using Qiskit's Pauli.evolve (Schrödinger frame).
    Drops global Pauli phase.
    """
    p = list(pauli_string)

    for inst in layer.data:
        if len(inst.qubits) != 2:
            continue

        q1 = inst.qubits[0]._index  # first qarg (e.g. CX control)
        q2 = inst.qubits[1]._index  # second qarg (e.g. CX target)
        gate = inst.operation

        # local qubit0=q1, qubit1=q2  -> label must be [q2][q1]
        p_local = Pauli(p[q2] + p[q1])

        p_evolved = p_local.evolve(gate, frame="s")
        lab = p_evolved.to_label().lstrip("+-i")  # drop phase

        # lab[0] is local qubit1 (=q2), lab[1] is local qubit0 (=q1)
        p[q2], p[q1] = lab[0], lab[1]

    return "".join(p)



def pauli_twirl_layers(qc, layers, num_samples=1):
    """
    Perform Pauli twirling around the 2Q Clifford part of each layer.

    Returns:
        {"layer0": [qc_sample_0, qc_sample_1, ...], ...}
    """
    num_qubits = qc.num_qubits
    twirled = {}

    for ell, layer in enumerate(layers):
        layer_samples = []

        for _ in range(num_samples):
            qc_twirl = qc.copy_empty_like()

            # --- (A) single-qubit gates FIRST (ideal dynamics)
            for inst in layer.data:
                if len(inst.qubits) == 1:
                    qc_twirl.append(inst.operation, inst.qubits)

            qc_twirl.barrier(label=f"1Q layer {ell}")

            # --- (B) sample Pauli
            P = [random.choice(PAULIS) for _ in range(num_qubits)]

            qc_twirl = apply_pauli_string(qc_twirl, P)
            qc_twirl.barrier(label=f"Twirl before 2Q layer {ell}")

            # --- (C) apply only 2Q Cliffords
            for inst in layer.data:
                if len(inst.qubits) == 2:
                    qc_twirl.append(inst.operation, inst.qubits)

            qc_twirl.barrier(label=f"2Q layer {ell}")

            # --- (D) apply conjugated Pauli
            P_conj = conjugate_pauli_2q(P, layer)
            qc_twirl = apply_pauli_string(qc_twirl, P_conj)
            qc_twirl.barrier(label=f"Twirl after 2Q layer {ell}")

            layer_samples.append(qc_twirl)

        twirled[f"layer{ell}"] = layer_samples

    return twirled


N=100
twirled_layers = pauli_twirl_layers(qc_bound, layers, num_samples=N)

print("One twirled sample:")
print(twirled_layers["layer0"][0].draw(fold=-1))

One twirled sample:
     ┌───┐┌─────────────┐┌────────────┐ 1Q layer 0 ┌───┐ Twirl before 2Q layer 0       2Q layer 0 ┌───┐ Twirl after 2Q layer 0 
q_0: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├─────░──────┤ Y ├────────────░──────────────■───────░──────┤ X ├───────────░────────────
     ├───┤├─────────────┤├────────────┤     ░      ├───┤            ░            ┌─┴─┐     ░      ├───┤           ░            
q_1: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├─────░──────┤ Z ├────────────░────────────┤ X ├─────░──────┤ Y ├───────────░────────────
     ├───┤├─────────────┤├────────────┤     ░      └───┘            ░            └───┘     ░      └───┘           ░            
q_2: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├─────░───────────────────────░──────────────■───────░──────────────────────░────────────
     ├───┤├─────────────┤├────────────┤     ░                       ░            ┌─┴─┐     ░                      ░            
q_3: ┤ H ├┤ Rz(-1.3606) ├┤ Rx(1.5808) ├─────░───────────────────────░────────────┤ X

## Sanity Check

For all the circuit instances the expectationvalue should be the same, we check that with a Statevectorsimulation.

In [140]:
from qiskit.quantum_info import Statevector
from qiskit import QuantumCircuit

qc_twirl = QuantumCircuit(qc_bound.num_qubits)

for ell, layer in enumerate(layers):
    twirled_layer = twirled_layers[f"layer{ell}"][0]
    qc_twirl.compose(twirled_layer, inplace=True)

psi_ref = Statevector.from_instruction(qc_bound)
psi_twirl = Statevector.from_instruction(qc_twirl)

print(f"original circuit: E = {np.real(psi_ref.expectation_value(Hamiltonian))}")
print(f"twirled circuit: E = {np.real(psi_twirl.expectation_value(Hamiltonian))}")

original circuit: E = -8.536285524802459
twirled circuit: E = -8.536285524802459


## Pauli–Lindblad noise model

### From discrete channels to generators

Instead of parameterizing $\Lambda$ directly, it is often more convenient to
parameterize a **generator** $\mathcal{L}$ and define the channel as

$$
\Lambda = e^{\mathcal{L}} .
$$

This form is familiar from open quantum systems:
if $\mathcal{L}$ generates a Markovian master equation

$$
\frac{d\rho(t)}{dt} = \mathcal{L}(\rho(t)),
$$

then the solution after a time step $\Delta t$ is

$$
\rho(\Delta t) = e^{\Delta t \mathcal{L}}(\rho(0)).
$$

In the circuit setting, we interpret **one circuit layer** as one fixed time step.
Writing the noise per layer as an exponential of a generator ensures that the resulting
map is **completely positive and trace preserving**.

---

### Lindblad generators built from Paulis

After twirling, it is natural to choose a generator that is itself diagonal in the
Pauli basis. This is achieved by using **Pauli jump operators**.

For a Pauli operator $P_k$, define the superoperator

$$
\mathcal{P}_k(\rho) = P_k \rho P_k .
$$

The Pauli–Lindblad generator is defined as

$$
\mathcal{L}
=
\sum_k \lambda_k \bigl( \mathcal{P}_k - \mathcal{I} \bigr),
\qquad
\lambda_k \ge 0 .
$$

Each term has a clear interpretation:

- $\mathcal{P}_k$ flips the state by conjugation with $P_k$.
- Subtracting the identity $\mathcal{I}$ ensures trace preservation.
- The coefficient $\lambda_k$ sets the **rate** at which this Pauli error occurs.

This is a special case of the general Lindblad form, with Hermitian jump operators
proportional to Pauli matrices.

---

### Why this generator is Pauli-diagonal

Consider how $\mathcal{L}$ acts on a Pauli operator $P_a$.

Conjugation by another Pauli either preserves or flips the sign:

- If $P_k$ commutes with $P_a$, then $P_k P_a P_k = P_a$.
- If $P_k$ anticommutes with $P_a$, then $P_k P_a P_k = -P_a$.

Therefore,

$$
(\mathcal{P}_k - \mathcal{I})(P_a)
=
\begin{cases}
0, & [P_k, P_a] = 0, \\
-2 P_a, & \{P_k, P_a\} = 0 .
\end{cases}
$$

Summing over all $k$ gives

$$
\mathcal{L}(P_a)
=
-2
\Bigl(
\sum_{k:\, \{P_k,P_a\}=0} \lambda_k
\Bigr)
P_a .
$$

Thus **every Pauli is an eigen-operator** of $\mathcal{L}$.

Exponentiating yields the channel action

$$
\Lambda(P_a)
=
\exp\!\Bigl(
-2 \sum_{k:\, \{P_k,P_a\}=0} \lambda_k
\Bigr)
P_a .
$$

This shows explicitly that the Pauli–Lindblad channel is Pauli-diagonal, and it
connects the generator parameters $\lambda_k$ directly to the Pauli fidelities.

---

### Relation to Pauli fidelities

Comparing with the twirled form

$$
\Lambda(P_a) = f_a P_a ,
$$

we identify

$$
f_a
=
\exp\!\Bigl(
-2 \sum_{k:\, \{P_k,P_a\}=0} \lambda_k
\Bigr).
$$

Hence:
- Pauli fidelities are **not independent parameters**.
- They are constrained by a small set of non-negative rates $\lambda_k$.
- This guarantees $0 < f_a \le 1$ automatically.

### Choosing the model terms (sparsity assumption)

In practice, we do not include all Paulis in the generator.
Instead, we restrict to terms that reflect the hardware noise structure:

- **Weight-1 terms**: $X_i, Y_i, Z_i$ on each qubit, capturing local decoherence and control errors.
- **Weight-2 terms**: $P_i \otimes P_j$ on qubit pairs $(i,j)$ that participate in entangling gates,
  capturing correlated errors introduced by two-qubit operations.

This choice defines the **Pauli–Lindblad model space**. The corresponding rates
$\{\lambda_k\}$ are the parameters to be learned experimentally.


In [141]:
from itertools import product, cycle, permutations

PAULIS_1Q = ["X", "Y", "Z"]
PAULIS_2Q = list(product(["X", "Y", "Z"], repeat=2))

def build_pauli_lindblad_terms(qc, backend, phys_qubits=None):
    """
    Build sparse Pauli–Lindblad model terms using BackendV2 target connectivity.
    """
    num_qubits = qc.num_qubits
    mapping = {phys: log for log, phys in enumerate(phys_qubits)}

    M = np.array(backend.coupling_map.get_edges())
    mask = np.isin(M[:, 0], phys_qubits) & np.isin(M[:, 1], phys_qubits)
    used_edges = np.unique(np.sort(M[mask], axis=1), axis=0)

    terms = []

    # --- 1Q terms ---
    for q in phys_qubits:
        q = mapping[q]
        for p in PAULIS_1Q:
            label = ["I"] * num_qubits
            label[q] = p
            terms.append("".join(label))

    # --- 2Q terms ---
    for (q1, q2) in used_edges:
        q1 = mapping[q1]
        q2 = mapping[q2]
        for p1, p2 in PAULIS_2Q:
            label = ["I"] * num_qubits
            label[q1] = p1
            label[q2] = p2
            terms.append("".join(label))

    return terms

from qiskit_ibm_runtime.fake_provider import FakeWashingtonV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

fake_backend = FakeWashingtonV2()
noise_model = NoiseModel.from_backend(fake_backend)
backend_noisy = AerSimulator(noise_model=noise_model)

phys_qubits = [3,4,5,15]

model_terms = build_pauli_lindblad_terms(qc_bound, fake_backend, phys_qubits)
print(f"Number of model terms: {len(model_terms)}")

Number of model terms: 39


In the next step we group the qubit wise commuting model terms for efficient measurement.

In [ ]:
NUM_BASES = 9

bases = [['I']*num_qubits for i in range(NUM_BASES)]
connectivity = fake_backend.coupling_map.graph.subgraph(phys_qubits)

for vertex in range(num_qubits):
    #copied from Fig. S3 in van den Berg
    orderings = {"XXXYYYZZZ":"XYZXYZXYZ",
                        "XXXYYZZZY":"XYZXYZXYZ",
                        "XXYYYZZZX":"XYZXYZXYZ",
                        "XXZYYZXYZ":"XYZXZYZYX",
                        "XYZXYZXYZ":"XYZZXYYZX"}
    
    children = connectivity.neighbors(vertex)
    predecessors = [c for c in children if c < vertex]

    match len(predecessors):
        #trivial if no predecessors
        case 0:
            cycp = cycle("XYZ")
            for i,_ in enumerate(bases):
                bases[i][vertex] = next(cycp)
        #Choose p1:"XXXYYYZZZ" and p2:"XYZXYZXYZ" if one predecessor
        case 1:
            pred, = predecessors
            #store permutation of indices so that predecessor has X,X,X,Y,Y,Y,Z,Z,Z
            _,bases = list(zip(*sorted(zip([p[pred] for p in bases], bases))))
            cycp = cycle("XYZ")
            for i,_ in enumerate(bases):
                bases[i][vertex] = next(cycp)
        case 2:
            pred0,pred1 = predecessors
            _,bases = list(zip(*sorted(zip([p[pred0] for p in bases], bases))))
            #list out string with permuted values of predecessor 2
            substr = [p[pred0] for p in bases]
            #match predecessor two with a permutation of example_orderings
            reordering = ""
            for perm in permutations("XYZ"):
                substr = "".join(["XYZ"[perm.index(p)] for p in substr])
                if substr in orderings:
                    current = orderings[substr] 
                    for i,p in enumerate(current):
                        bases[i][vertex] = p
                    break
        case _: #processor needs to have connectivity so that there are <= 2 predecessors
            raise Exception("Three or more predecessors encountered")

print("Measurement bases:", ["".join(b) for b in bases])
bases = [Pauli("".join(string[::-1])) for string in bases]

## Simultaneous single measurements and degeneracy lifting

At first sight, estimating the fidelity of a Pauli–Lindblad model term seems
straightforward. After Pauli twirling, each Pauli operator $P$ is an
eigen-operator of the noise channel, and one might expect that its fidelity
can be extracted by repeatedly running the circuit and measuring the decay
of $\langle P \rangle$.

This intuition is *only correct if the noise channel acts in isolation*.
In practice, each noisy layer has the form
$$
\tilde{\mathcal U} = \mathcal U \circ \Lambda ,
$$
where $\mathcal U(\rho) = U\rho U^\dagger$ is a Clifford layer (typically a
collection of CX or CZ gates) and $\Lambda$ is the Pauli-twirled noise.

---

### Why the naive decay picture fails

To see the problem, switch to the Heisenberg picture.
Suppose we attempt to track the decay of a Pauli operator $P$.
After one layer we have
$$
P \;\xleftarrow{\;\mathcal U\;}\; U^\dagger P U .
$$
Because $U$ is a Clifford operator, $U^\dagger P U$ is *another* Pauli.
On the next repetition, the noise acts on this new Pauli instead of the
original one.

As a result, the Pauli operator whose expectation value we are tracking
**changes from repetition to repetition**.

---

### Example: CZ-induced degeneracy

Consider a CZ gate acting on qubits $i$ and $j$. One has
$$
\mathrm{CZ}^\dagger (I_i \otimes X_j)\,\mathrm{CZ}
=
Z_i \otimes X_j .
$$
If we try to measure the decay of $I_i X_j$, then successive repetitions
of the noisy CZ layer produce the sequence
$$
I_i X_j \;\longleftrightarrow\; Z_i X_j \;\longleftrightarrow\; I_i X_j \;\cdots
$$

After $2m$ repetitions, the measured expectation value behaves as
$$
\langle I_i X_j \rangle_{2m}
=
\alpha \,(f_{I_i X_j} f_{Z_i X_j})^{m}.
$$
Only the *product* of fidelities appears. The individual fidelities
$f_{I_i X_j}$ and $f_{Z_i X_j}$ are never observed separately.

---

### Degeneracy in the linear inversion problem

In the Pauli–Lindblad model, fidelities are related to Lindblad rates
$\lambda$ via
$$
-\tfrac{1}{2}\log f = M \lambda ,
$$
where the matrix $M$ is determined by Pauli commutation relations.

When Clifford conjugation forces two Pauli operators to always appear as a
pair (as in the example above), the corresponding columns of $M$ are
identical. This creates a **degeneracy**:
the matrix $M$ loses rank and cannot be inverted.

This degeneracy arises whenever:
- a Pauli and its Clifford conjugate are distinct, and
- both are included as separate terms in the sparse model.

---

### Degeneracy lifting via additional measurement bases

The degeneracy is not fundamental; it can be lifted by performing a small number of additional measurements in
different single-qubit bases.

Key observations:
- Pauli terms acting on *disjoint gates* do not interfere and can be measured
  simultaneously.
- For a two-qubit Clifford gate, two single-qubit Pauli operators are
  stabilized by the Clifford and do not mix.
- The remaining four single-qubit Pauli terms require additional basis
  choices to separate their contributions.
- In addition, two weight-two Pauli terms must be measured to fully lift the
  degeneracy.

Altogether, **six measurement bases per gate** suffice to uniquely identify
all model parameters.


In [199]:
def get_expectation(pauli, counts):
    """
    Expectation value of a Pauli string from Z-basis counts after the proper basis rotation.
    Assumption: X/Y were rotated to Z before measurement (H or Sdg+H).

    Qiskit convention:
    - Pauli label: left = qubit n-1, right = qubit 0
    - Bitstring:   left = qubit n-1, right = qubit 0
    """
    label = pauli.to_label() if hasattr(pauli, "to_label") else str(pauli)
    shots = sum(counts.values())
    if shots == 0:
        return 0.0

    exp = 0.0
    for bitstring, cnt in counts.items():
        parity = 0
        for pos, p in enumerate(label):         # pos=0 ↔ qubit n-1 … pos=n-1 ↔ qubit 0
            if p != 'I':
                parity ^= (bitstring[pos] == '1')
        exp += (1.0 if parity == 0 else -1.0) * cnt

    return exp / shots

def pauli_exp_dict_local(groups, counts):
    pauli_expectations = []

    for idx, group in enumerate(groups):
        group_expectation = {}
        for pauli_str in group:
            pauli = Pauli(pauli_str)
            exp_val = get_expectation(pauli=pauli, counts=counts[idx])
            group_expectation[pauli_str] = exp_val
        pauli_expectations.append(group_expectation)

    merged = {}
    for d in pauli_expectations:
        merged.update(d)
    return merged

from qiskit.quantum_info import Pauli
import networkx as nx

def locally_commutable(p1, p2):
    """
    Check whether two Pauli strings are locally measurable:
    no qubit has different non-identity Paulis (e.g., X and Y) at the same time.
    """
    for a, b in zip(p1, p2):
        if a == 'I' or b == 'I':
            continue
        if a != b:
            return False  # different non-I terms → not locally measurable at once
    return True

def group_paulis(pauli_strings):

    # ===== Normal (non-trivial) case with graph coloring =====
    G = nx.Graph()
    G.add_nodes_from(range(len(pauli_strings)))

    # Edge if they are NOT jointly measurable (conflict graph)
    for i in range(len(pauli_strings)):
        for j in range(i + 1, len(pauli_strings)):
            if not locally_commutable(pauli_strings[i], pauli_strings[j]):
                G.add_edge(i, j)

    # Graph coloring
    coloring = nx.coloring.greedy_color(G, strategy="largest_first")

    # Group by colors
    groups = {}
    for idx, color in coloring.items():
        groups.setdefault(color, []).append(pauli_strings[idx])

    return list(groups.values())

def determine_measurement_bases(pauli_groups: list[list[str]]) -> list[list[str]]:
    """
    Determine measurement bases for a list of Pauli groups.
    Returns: list of bases (each a list of 'X', 'Y', 'Z', 'I').
    """
    bases = []
    for group in pauli_groups:
        num_qubits = len(group[0])
        basis = ['Z'] * num_qubits  # Default: measure in Z

        for qubit in range(num_qubits):
            paulis_on_qubit = set(
                p[num_qubits - 1 - qubit] for p in group if p[num_qubits - 1 - qubit] != 'I'
            )

            if not paulis_on_qubit:
                basis[qubit] = 'I'
            elif paulis_on_qubit == {'Z'}:
                basis[qubit] = 'Z'
            elif paulis_on_qubit == {'X'}:
                basis[qubit] = 'X'
            elif paulis_on_qubit == {'Y'}:
                basis[qubit] = 'Y'
            else:
                raise ValueError(
                    f"Incompatible Paulis on qubit {qubit}: {paulis_on_qubit}"
                )

        bases.append(basis)

    return bases

def apply_measurement_bases(base_circuit, measurement_bases):
    # Creates the basis transformation from a measurment basis to computational basis and applies the measurement.
    circuits=[]

    for basis in measurement_bases:
        qc = base_circuit.copy()

        for qubit, b in enumerate(basis):
            if b == 'X':
                qc.h(qubit)
            elif b == 'Y':
                qc.sdg(qubit)
                qc.h(qubit)
            elif b == 'Z':
                pass
            elif b == 'I':
                pass
            else:
                raise ValueError(f"Undefined measurement basis: {b} for qubit {qubit}")

        qc.measure_all()
        circuits.append(qc)

    return circuits

In [200]:
from collections import defaultdict
from qiskit.primitives import BackendSamplerV2

groups = group_paulis(model_terms)
measurement_bases = determine_measurement_bases(groups)
layers_with_meas = {}
for i in range(len(twirled_layers)):
    layers_with_meas[f"layer{i}"] = []
    for sample in twirled_layers[f"layer{i}"]:
            layers_with_meas[f"layer{i}"].extend(apply_measurement_bases(sample, measurement_bases))

shots=1000
sampler = BackendSamplerV2(backend = backend_noisy, options={"default_shots":shots})

def collect_expectations(circuits,backend,shots,model_terms):
    """
    Run circuits, return expectation values grouped by type.
    """
    
    expvals = {"double": defaultdict(list),"single": defaultdict(list),}
    result = sampler.run(circuits).result()
    counts_dict = [r.data.meas.get_counts() for r in result]

    for i, circ in enumerate(circuits):
        counts = counts_dict[i]
        meta = circ.metadata
        typ  = meta["type"]

        # expectation values of all model terms
        for P in model_terms:
            exp = get_expectation(Pauli(P), counts)
            expvals[typ][P].append(exp)

    return expvals

def extract_fidelities(expvals,depths,degeneracy_pairs):
    """
    Combine double + single experiments to extract individual fidelities.
    """
    fidelities = {}

    # --- 1) handle non-degenerate Paulis ---
    for P, vals in expvals["double"].items():
        if P not in degeneracy_pairs:
            y = np.log(np.abs(vals))
            x = np.array(depths)
            fidelities[P] = np.exp(np.polyfit(x, y, 1)[0])
    
    # --- 2) handle degenerate pairs ---
    for P, Pp in degeneracy_pairs:
        # product from double
        y = np.log(np.abs(expvals["double"][P]))
        x = np.array(depths)
        prod = np.exp(np.polyfit(x, y, 1)[0] * 2)

        # individual from single
        sP  = np.mean(expvals["single"][P])
        sPp = np.mean(expvals["single"][Pp])

        fidelities[P]  = np.abs(sP)
        fidelities[Pp] = prod / fidelities[P]

    return fidelities

In [202]:
layer0_circuits = layers_with_meas["layer0"]
result = sampler.run(layer0_circuits).result()
counts_dict = [r.data.meas.get_counts() for r in result]

expvals = pauli_exp_dict_local(groups, counts_dict)

print(expvals)

{'XXII': 0.154, 'IXXI': 0.048, 'IXIX': 0.002, 'IXII': 0.032, 'XIII': 0.158, 'IIXI': 0.208, 'IIIX': 0.002, 'XYII': -0.012, 'IYXI': -0.036, 'IYIX': 0.054, 'IYII': -0.012, 'XZII': -0.168, 'IZXI': -0.212, 'IZIX': -0.042, 'IZII': -0.92, 'YXII': 0.03, 'IXYI': -0.002, 'IXIY': 0.04, 'YIII': 0.04, 'IIYI': -0.004, 'IIIY': -0.042, 'YYII': 0.194, 'IYYI': 0.044, 'IYIY': 0.014, 'YZII': -0.034, 'IZYI': 0.006, 'IZIY': 0.004, 'ZXII': -0.018, 'IXZI': 0.016, 'IXIZ': -0.024, 'ZIII': 0.864, 'IIZI': 0.926, 'IIIZ': -0.966, 'ZYII': 0.03, 'IYZI': 0.016, 'IYIZ': -0.028, 'ZZII': -0.864, 'IZZI': -0.854, 'IZIZ': 0.902}


In [203]:
depths = [1, 2, 4, 8]
circuits = []

for d in depths:
    circ = QuantumCircuit(qc_bound.num_qubits)
    for _ in range(d):
        circ.compose(layers[0], inplace=True)
    circ.measure_all()
    circuits.append(circ)

result = sampler.run(circuits).result()
counts_dict = [r.data.meas.get_counts() for r in result]

expvals = []
P = Pauli(model_terms[0])
for i, d in enumerate(depths):
    counts = counts_dict[i]
    expvals.append(get_expectation(P, counts))

print(list(zip(depths, expvals)))

import numpy as np

def extract_fidelity(depths, expvals):
    y = np.log(np.abs(expvals))
    x = np.array(depths)
    slope, _ = np.polyfit(x, y, 1)
    return np.exp(slope)



[(1, 0.888), (2, -0.818), (4, 0.732), (8, 0.256)]
